In [ ]:
import nltk
from nltk.corpus import stopwords
import csv
from nltk.tag import pos_tag #used for proper nouns
from nltk.tokenize import word_tokenize, sent_tokenize #For tokens
import pandas as pd
from nltk.stem import PorterStemmer

In [ ]:
filename="C:\\Users\\saura\\OneDrive\\Desktop\\unlabelled documents\\12.txt"
f=open( (filename), "r")
text=f.read() 
f.close()


In [ ]:
sent_tokens=nltk.sent_tokenize(text)
word_tokens=nltk.word_tokenize(text) #Tokenizing the words
word_tokens_lower= [word.lower() for word in word_tokens]
stopWords=list(set(stopwords.words ("english")))
word_tokens_refined=[x for x in word_tokens_lower if x not in stopWords] #Removing the stop words


In [ ]:
# The cue phrases for covid
QPhrases=["covid","covid-19","lockdown","quarantine","isolation","PPE","sanitiser","panic","distance","outbreak","case","spread","confirm","case","positive","corona","asymptomatic","masks","corona","virus","test","negative","pandemic","infected","WHO","vaccine","incidentally", "example", "anyway", "furthermore","according",  "first", "second", "then", "now", "thus", "moreover", "therefore", "hence", "lastly", "finally", "summary"]
cue_phrases={}
for sentence in sent_tokens:
    cue_phrases[sentence]=0
    word_tokens=nltk.word_tokenize(sentence)
    for word in word_tokens:
        if word.lower() in QPhrases:
            cue_phrases[sentence] +=1
maximum_frequency=max(cue_phrases.values())
for i in cue_phrases.keys():
    try:
        cue_phrases[i]=cue_phrases[i]/maximum_frequency
        cue_phrases[i]=round(cue_phrases[i],2)
    except ZeroDivisionError:
        x=0


In [ ]:
# Based on number of words matching the heading
heading={}
head_word=nltk.word_tokenize(sent_tokens[0])
heading[sent_tokens[0]]=0
for sentence in sent_tokens[1:]:
    heading[sentence]=0
    word=nltk.word_tokenize(sentence)
    for every in head_word:
        if every in word:
            heading[sentence]+=1
        
maximum_frequency=max(heading.values())
heading[sent_tokens[0]]=maximum_frequency

for k in heading.keys():
    try :
        heading[k]=heading[k]/maximum_frequency
        heading[k]=round(heading[k],2)
    except ZeroDivisionError:
        x=0            
    

In [ ]:
# If there is numerical value present in the sentence or not
numeric_data={}
for sentence in sent_tokens:
    numeric_data[sentence]=0
    word_tokens=nltk.word_tokenize(sentence)
    for word in word_tokens:
       if word.isdigit():
            numeric_data[sentence] +=1
maximum_frequency=max(numeric_data.values())
for i in numeric_data.keys():
    try:
        numeric_data[i]=(numeric_data[i]/maximum_frequency)
        numeric_data[i]= round(numeric_data[i], 1)
    except ZeroDivisionError:
        x=0

In [ ]:
# Based on sentence length
sent_len_score={}
for sentence in sent_tokens:
    sent_len_score[sentence]=0
    word_tokens=nltk.word_tokenize(sentence)
    if len(word_tokens) in range(0,10):
        sent_len_score[sentence]=1-0.64*(10-len(word_tokens))
    elif len(word_tokens) in range(11,20):#Optimal length of the word
        sent_len_score[sentence]=1
    else:
        sent_len_score[sentence]=1-(0.05)*(len(word_tokens)- 20)
for i in sent_len_score.keys():
    sent_len_score[i]=round(sent_len_score[i],2)

In [ ]:
# Based on the position of the sentence
sentence_position={}
d=1
no_of_sent=len(sent_tokens)
for i in range(no_of_sent):
     a=1/d
     b=1/(no_of_sent-d+1)
     sentence_position[sent_tokens[d-1]]=max(a,b)
     d=d+1
for k in sentence_position.keys():
     sentence_position[k]=round(sentence_position[k],2)
print(sentence_position)

In [ ]:
# Based on whether the  number of upper case values
upper_case={}
for sentence in sent_tokens:
   upper_case[sentence]=0
   word_tokens=nltk.word_tokenize(sentence)
   for k in word_tokens:
       if k.isupper():
          upper_case[sentence] +=1
maximum_frequency=max(upper_case.values())
for k in upper_case.keys():
   try:
       upper_case[k]=(upper_case[k]/maximum_frequency)
       upper_case[k]=round(upper_case[k],2)
   except ZeroDivisionError:
       x=0
print(upper_case)

In [ ]:
# Based on the number of the proper nouns

proper_noun={}
for sentence in sent_tokens:
    tagged_sent=pos_tag(sentence.split())
    propernouns=[word for word,pos in tagged_sent if pos == 'NNP']
    proper_noun[sentence]=len(propernouns)
maximum_frequency =max(proper_noun.values())
for k in proper_noun.keys():
    try:
        proper_noun[k]=(proper_noun[k]/maximum_frequency)
        proper_noun[k]=round(proper_noun[k],2)
    except ZeroDivisionError:
        X=0
print(proper_noun)

In [ ]:
# Calculating the total score of each sentence
total_score={}
for k in cue_phrases.keys():
    total_score[k]=cue_phrases[k]+numeric_data[k]+sent_len_score[k]+sentence_position[k]+upper_case[k]+proper_noun[k]
print(total_score)

In [ ]:
# Calculating the average value
sumvalues=0
for sentence in total_score:
    sumvalues += total_score[sentence]
average=int(sumvalues/len(total_score))


In [ ]:

summary=''
for sentence in sent_tokens:
    if (sentence in total_score) and (total_score[sentence]>1.5*(average)):
        summary +="  "+sentence
print(summary)